# Netflix Customer Churn - Data Preprocessing
**Notebook outline:**
1. Setup and data loading
2. Feature selection and redundancy removal
3. Feature engineering
4. Train-test split
5. Encoding and feature scaling
6. Exporting processed datasets

## 1. Setup and data loading

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

In [2]:
random_state = 42

raw_data_path = Path("../data/raw/netflix_customer_churn.csv")
processed_data_path = Path("../data/processed")
models_path = Path("../models")

In [3]:
assert raw_data_path.exists(), print(f"Path does not exist: {raw_data_path}")
print(f"Path exists: {raw_data_path}")

Path exists: ..\data\raw\netflix_customer_churn.csv


In [4]:
df = pd.read_csv(raw_data_path)

assert df.isnull().sum().sum() == 0, "Data contains missing values"
print("Data contains no missing values")

df.head(10)

Data contains no missing values


,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action
5,d8079475-5be7-47e9-8782-ceb7ff61395e,58,Female,Standard,13.80,26,Oceania,Mobile,13.99,0,Debit Card,3,0.51,Action
6,8e63450a-13d6-4e83-bbb5-6aebde9152cb,48,Other,Basic,13.83,20,Asia,TV,8.99,0,Gift Card,5,0.66,Romance
7,02387681-8c42-462a-807a-de0168c73b38,51,Male,Basic,14.30,56,Europe,Mobile,8.99,1,Gift Card,1,0.25,Action
8,0bcaad0c-545c-4ee1-85a6-75e165f39361,45,Other,Basic,9.98,10,Asia,Mobile,8.99,0,PayPal,3,0.91,Romance
9,eae6439e-8cdf-4258-ab49-c493925b927a,32,Other,Premium,2.22,34,Europe,TV,17.99,1,Debit Card,1,0.06,Drama


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5000 non-null   str    
 1   age                     5000 non-null   int64  
 2   gender                  5000 non-null   str    
 3   subscription_type       5000 non-null   str    
 4   watch_hours             5000 non-null   float64
 5   last_login_days         5000 non-null   int64  
 6   region                  5000 non-null   str    
 7   device                  5000 non-null   str    
 8   monthly_fee             5000 non-null   float64
 9   churned                 5000 non-null   int64  
 10  payment_method          5000 non-null   str    
 11  number_of_profiles      5000 non-null   int64  
 12  avg_watch_time_per_day  5000 non-null   float64
 13  favorite_genre          5000 non-null   str    
dtypes: float64(3), int64(4), str(7)
memory usage: 547.0

In [6]:
df.columns.tolist()

['customer_id',
 'age',
 'gender',
 'subscription_type',
 'watch_hours',
 'last_login_days',
 'region',
 'device',
 'monthly_fee',
 'churned',
 'payment_method',
 'number_of_profiles',
 'avg_watch_time_per_day',
 'favorite_genre']

In [7]:
df['churned'].value_counts(normalize=True)

churned
1    0.503
0    0.497
Name: proportion, dtype: float64

> **Check:** the raw dataset loads cleanly with 5,000 rows, 14 columns, zero missing values and the expected target balance (~50.3% churned), matching the EDA baseline.

## 2. Feature selection and redundancy removal

In [15]:
df.drop(columns=['customer_id'], inplace=True)

KeyError: "['customer_id'] not found in axis"

In [17]:
df.drop(columns=['subscription_type'], inplace=True)

In [20]:
df.drop(columns=['avg_watch_time_per_day'], inplace=True)

In [24]:
df.drop(columns=['gender', 'region', 'device', 'favorite_genre'], inplace=True)

In [26]:
df.drop(columns=['age'], inplace=True)

In [29]:
df_clean = df.copy()

In [31]:
df_clean.head()

,watch_hours,last_login_days,monthly_fee,churned,payment_method,number_of_profiles
0,14.73,29,8.99,1,Gift Card,1
1,0.70,19,13.99,1,Gift Card,5
2,16.32,10,13.99,0,Crypto,2
3,4.51,12,17.99,1,Crypto,2
4,1.89,13,13.99,1,Crypto,2


> **Decision:** 8 columns are removed prior to feature engineering based on the EDA findings:
> - `customer_id` is a unique identifier with no predictive value.
> - `subscription_type` is a 1:1 duplicate of `monthly_fee`, and `avg_watch_time_per_day` is deterministically derived from `watch_hours` and `last_login_days`. Dropping both prevents redundancy.
> - `gender`, `region`, `device`, `favorite_genre` and `age` showed flat churn distributions and no meaningful predictive signal in EDA, so removing them reduces noise.

## 3. Feature engineering

In [32]:
def add_features(df):
    df = df.copy()

    df["log_watch_hours"] = np.log1p(df["watch_hours"].clip(lower=0))
    df["inactive_30d"] = (df["last_login_days"] > 30).astype(int)
    df["profiles_4plus"] = (df["number_of_profiles"] >= 4).astype(int) 
    df["risky_payment"] = (df["payment_method"].isin(["Crypto", "Gift Card"])).astype(int)

    return df

df_featured = add_features(df_clean)

df_featured.head()

,watch_hours,last_login_days,monthly_fee,churned,payment_method,number_of_profiles,log_watch_hours,inactive_30d,profiles_4plus,risky_payment
0,14.73,29,8.99,1,Gift Card,1,2.755570,0,0,1
1,0.70,19,13.99,1,Gift Card,5,0.530628,0,1,1
2,16.32,10,13.99,0,Crypto,2,2.851862,0,0,1
3,4.51,12,17.99,1,Crypto,2,1.706565,0,0,1
4,1.89,13,13.99,1,Crypto,2,1.061257,0,0,1


> **Decision:** four domain-informed features are added to capture non-linear patterns identified in EDA:
> - `log_watch_hours` (`log1p(watch_hours)`) compresses the right-skewed viewing tail (up to 110h) for linear and distance-based models while keeping raw `watch_hours` available for tree models.
> - `inactive_30d` captures the 30-day inactivity cliff, where churn jumps from 17-35% to 73-78%.
> - `profiles_4plus` captures the step-change at four profiles, where churn drops from ~57-59% (1-3 profiles) to ~38-41% (4-5 profiles).
> - `risky_payment` groups non-recurring payment methods (`Crypto` at 59.7% churn and `Gift Card` at 57.8%) versus recurring billing methods (43.6-47.1%).

## 4. Train-test split

In [41]:
y = df_featured["churned"]
X = df_featured.drop(columns=["churned"])

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=random_state, 
    stratify=y
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape} \n")
print(f"Churned distribution in: \n - train set: {y_train.mean():.2%} \n - test set: {y_test.mean():.2%}")

X_train shape: (4000, 9), X_test shape: (1000, 9) 

Churned distribution in: 
 - train set: 50.30% 
 - test set: 50.30%


> **Decision:** the dataset is split 80/20 into training (4,000 rows) and test (1,000 rows) sets with `stratify=y` and `random_state=42` before fitting any encoder or scaler. Stratification preserves the ~50.3% churn proportion in both splits, and splitting early prevents data leakage from `X_test`.

## 5. Encoding and feature scaling

In [ ]:
num_cols = [
    "watch_hours", 
    "last_login_days",
    "monthly_fee",
    "number_of_profiles",
    "log_watch_hours"
]

cat_cols = [
    "payment_method"
]

bin_cols = [
    "inactive_30d",
    "profiles_4plus",
    "risky_payment"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore"
                ), cat_cols),
        ("bin", "passthrough", bin_cols),
    ]
)

X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)

col_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_arr, columns=col_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_arr, columns=col_names, index=X_test.index)

print("Nulls in X_train_processed:", X_train_processed.isnull().sum().sum())
print("Nulls in X_test_processed:", X_test_processed.isnull().sum().sum())



Nulls in X_train_processed: 0
Nulls in X_test_processed: 0


> **Decision:** a `ColumnTransformer` fitted exclusively on `X_train` applies type-specific transformations without leakage:
> - Continuous features (`watch_hours`, `last_login_days`, `monthly_fee`, `number_of_profiles`, `log_watch_hours`) are scaled with `RobustScaler` to limit sensitivity to skew and outliers.
> - `payment_method` is encoded via `OneHotEncoder(drop="first", handle_unknown="ignore")`, producing 4 binary indicators without dummy-variable collinearity.
> - Engineered binary flags (`inactive_30d`, `profiles_4plus`, `risky_payment`) are passed through unscaled (`passthrough`) to preserve their 0/1 interpretation.

## 6. Exporting processed datasets

In [ ]:
processed_data_path.mkdir(parents=True, exist_ok=True)

X_train_processed.to_csv(processed_data_path / "X_train.csv", index=False)
X_test_processed.to_csv(processed_data_path / "X_test.csv", index=False)

y_train.to_csv(processed_data_path / "y_train.csv", index=False)
y_test.to_csv(processed_data_path / "y_test.csv", index=False)

print(f"Processed data saved to: {processed_data_path}")

Processed data saved to: ..\data\processed
